# Understanding Long Short-Term Memory Networks (LSTMs)

Long Short-Term Memory Networks are a special type of RNN designed to capture long-term dependencies in sequential data by using a more complex hidden state structure.

## LSTM Gates and Their Functions

For each time step $t$, the LSTM updates its cell state $c_t$ and hidden state $h_t$ using the current input $x_t$, the previous cell state $c_{t-1}$, and the previous hidden state $h_{t-1}$. The LSTM architecture consists of several gates that control the flow of information.

---

## Forget Gate $f_t$

This gate decides what information to discard from the cell state. It looks at the previous hidden state $h_{t-1}$ and the current input $x_t$, and outputs a number between $0$ and $1$ for each value in the cell state.

A value close to $1$ means **keep the information**, while a value close to $0$ means **forget the information**.

$$
f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)
$$

---

## Input Gate $i_t$

This gate decides which new information will be stored in the cell state. It consists of two parts:

1. A **sigmoid layer** that determines which values should be updated.
2. A **tanh layer** that generates candidate values $\tilde{c}_t$ that could be added to the cell state.

$$
i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)
$$

$$
\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)
$$

---

## Cell State Update $c_t$

This step updates the previous cell state $c_{t-1}$ to produce the new cell state $c_t$.

The old state is multiplied by the forget gate output, then the new candidate values are added after being scaled by the input gate.

$$
c_t = f_t \circ c_{t-1} + i_t \circ \tilde{c}_t
$$

Where $\circ$ denotes **element-wise multiplication**.

---

## Output Gate $o_t$

This gate determines what part of the cell state will be output as the hidden state.

First, a sigmoid layer determines which parts of the cell state to output. Then the result is multiplied by the $\tanh$ of the cell state.

$$
o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)
$$

$$
h_t = o_t \circ \tanh(c_t)
$$

---

## Parameters

- $W_f, W_i, W_c, W_o$ are the **weight matrices** for the forget, input, candidate cell, and output gates.
- $b_f, b_i, b_c, b_o$ are the **bias vectors**.
- $\sigma$ is the **sigmoid activation function**.
- $\circ$ denotes **element-wise multiplication**.

---

# Implementation Steps

1. **Initialization**

Start with the initial cell state $c_0$ and hidden state $h_0$.

2. **Sequence Processing**

For each input $x_t$ in the sequence:

- Compute the forget gate $f_t$
- Compute the input gate $i_t$
- Compute candidate cell state $\tilde{c}_t$
- Update the cell state $c_t$
- Compute the output gate $o_t$
- Compute the hidden state $h_t$

3. **Final Output**

After processing all inputs, the final hidden state $h_T$ (where $T$ is the sequence length) contains information from the entire sequence.

---

# Example Calculation

### Given

- Inputs:  
  $x_1 = 1.0$, $x_2 = 2.0$, $x_3 = 3.0$

- Initial states:  
  $c_0 = 0.0$, $h_0 = 0.0$

- Simplified weights (for demonstration):  

$$
W_f = W_i = W_c = W_o = 0.5
$$

- Bias values:

$$
b_f = b_i = b_c = b_o = 0.1
$$

---

## First Time Step ($t = 1$)

Forget gate:

$$
f_1 = \sigma(0.5 \times 1.0 + 0.1) = 0.6487
$$

Input gate:

$$
i_1 = \sigma(0.5 \times 1.0 + 0.1) = 0.6487
$$

Candidate cell state:

$$
\tilde{c}_1 = \tanh(0.5 \times 1.0 + 0.1) = 0.5370
$$

Cell state update:

$$
c_1 = f_1 \times 0.0 + i_1 \times \tilde{c}_1
$$

$$
c_1 = 0.6487 \times 0 + 0.6487 \times 0.5370 = 0.3484
$$

Output gate:

$$
o_1 = \sigma(0.5 \times 1.0 + 0.1) = 0.6487
$$

Hidden state:

$$
h_1 = o_1 \times \tanh(c_1)
$$

$$
h_1 = 0.6487 \times \tanh(0.3484) = 0.2169
$$

---

## Next Time Steps

### $t = 2$

Repeat the same steps using:

- $x_2 = 2.0$
- previous states $h_1$ and $c_1$

### $t = 3$

Repeat the same steps using:

- $x_3 = 3.0$
- previous states $h_2$ and $c_2$

The final hidden state $h_3$ represents the output after processing the full sequence.

---

# Applications

LSTMs are widely used in tasks involving sequential data, including:

- **Machine translation**
- **Speech recognition**
- **Time series forecasting**
- **Natural language processing**
- **Text generation**

They are particularly powerful when **long-term dependencies** must be captured across sequences.

[Problem Desc](https://www.deep-ml.com/problems/59?from=Deep%20Learning)

In [7]:
import numpy as np

class LSTM:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Initialize weights and biases
        self.Wf = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wi = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wc = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wo = np.random.randn(hidden_size, input_size + hidden_size)
        
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bc = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))
        
    def forward(self, x, initial_hidden_state, initial_cell_state):
        """
        Processes a sequence of inputs and returns the hidden states, final hidden state, and final cell state.
        """
        def sigmoid(z):
            return 1 / (1 + np.exp(-z))
        
        h = initial_hidden_state
        c = initial_cell_state
        outputs = []
        
        for x_i in x:
            x_i = x_i.reshape(-1, 1)
            
            combined = np.concatenate((h, x_i), axis=0)
            forget_gate = sigmoid(self.Wf @ combined + self.bf)
            
            input_gate = sigmoid(self.Wi @ combined + self.bi)
            candidate_cell_gate = np.tanh(self.Wc @ combined + self.bc)
            
            output_gate = sigmoid(self.Wo @ combined + self.bo)
            
            c = forget_gate * c + input_gate * candidate_cell_gate
            h = output_gate * np.tanh(c)
            
            outputs.append(h)
            
        return outputs, h, c

In [8]:
# Test Case 1
import numpy as np

input_sequence = np.array([[1.0], [2.0], [3.0]])
initial_hidden_state = np.zeros((1, 1))
initial_cell_state = np.zeros((1, 1))

lstm = LSTM(input_size=1, hidden_size=1)
# Set weights and biases for reproducibility
lstm.Wf = np.array([[0.5, 0.5]])
lstm.Wi = np.array([[0.5, 0.5]])
lstm.Wc = np.array([[0.3, 0.3]])
lstm.Wo = np.array([[0.5, 0.5]])
lstm.bf = np.array([[0.1]])
lstm.bi = np.array([[0.1]])
lstm.bc = np.array([[0.1]])
lstm.bo = np.array([[0.1]])

outputs, final_h, final_c = lstm.forward(input_sequence, initial_hidden_state, initial_cell_state)

print(final_h)

# Expected Output:
# [[0.73698596]]

[[0.73698596]]


In [9]:
# Test Case 2
import numpy as np

input_sequence = np.array([[0.1, 0.2], [0.3, 0.4]])
initial_hidden_state = np.zeros((2, 1))
initial_cell_state = np.zeros((2, 1))

lstm = LSTM(input_size=2, hidden_size=2)
# Set weights and biases for reproducibility
lstm.Wf = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm.Wi = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm.Wc = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm.Wo = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm.bf = np.array([[0.1], [0.2]])
lstm.bi = np.array([[0.1], [0.2]])
lstm.bc = np.array([[0.1], [0.2]])
lstm.bo = np.array([[0.1], [0.2]])

outputs, final_h, final_c = lstm.forward(input_sequence, initial_hidden_state, initial_cell_state)

print(final_h)

# Expected Output:
# [[0.16613133], [0.40299449]]

[[0.16613133]
 [0.40299449]]
